In [1]:
%pip install langchain langchain-community
%pip install langchain-text-splitters
%pip install pydantic

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:

from langchain_text_splitters import RecursiveCharacterTextSplitter
from pydantic import BaseModel, Field
from typing import List, Optional, Any

class JobInput(BaseModel):
    serial_no: int
    media_site: Optional[str] = Field(None, alias="Media site")
    company_name: str = Field(..., alias="Company name")
    company_logo: Optional[str] = Field(None, alias="Company logo")
    title: Optional[str] = Field(None, alias="Title")
    catchphrase: Optional[str] = None
    salary_type: Optional[str] = Field(None, alias="Salary type")
    salary: Optional[str] = None  
    salary_details: Optional[str] = Field(None, alias="Salary details")
    employment_type: Optional[str] = Field(None, alias="Employment Type")
    job_industry: Optional[str] = Field(None, alias="Job Industry")
    job_category: Optional[str] = Field(None, alias="Job Category")
    social_insurances: Optional[str] = Field(None, alias="Social insurances")
    job_benefits_details1: Optional[str] = Field(None, alias="Job Benefits Details1")
    job_benefits_details2: Optional[str] = Field(None, alias="Job Benefits Details2")
    holidays_leaves_details: Optional[str] = Field(None, alias="Holidays & Leaves Details")
    description: Optional[str] = Field(None, alias="Description")
    requirements: Optional[str] = None
    requirements_summary: Optional[str] = Field(None, alias="Requirements summary")
    service_form: Optional[str] = Field(None, alias="Service Form")
    working_hours: Optional[str] = Field(None, alias="Working hours")
    one_day_work_details: Optional[str] = Field(None, alias="One day work details")
    nearest_station: Optional[str] = Field(None, alias="Nearest Station")
    nearest_station_access: Optional[str] = Field(None, alias="Nearest station access")
    selection_flow: Optional[str] = Field(None, alias="Selection flow")
    recruiter_message: Optional[str] = Field(None, alias="Recruiter message")
    postal_code: Optional[str] = Field(None, alias="Postal Code")
    address_details: Optional[str] = Field(None, alias="Address details")
    google_map_url: Optional[str] = Field(None, alias="Google Map Url")
    trial_period_duration: Optional[str] = Field(None, alias="Trial period duration")
    trial_period_details: Optional[str] = Field(None, alias="Trial period details")
    trial_period_salary: Optional[str] = Field(None, alias="Trial period salary")
    trial_period_working_hours: Optional[str] = Field(None, alias="Trial period working hours")
    tag: List[str] = Field([], alias="Tag")
    image_data: Optional[str] = Field(None, alias="image data")

    class Config:
        populate_by_name = True








def chunking(job):
    """
    Convert a single job object into multiple semantic chunks for RAG.
    This version does NOT use a safe() helper; all fields are directly concatenated.
    """

    # -------------------------------
    # Job data extraction (image_data ignored)
    # -------------------------------
    data = {
        "serial_no": job.serial_no,
        "media_site": job.media_site,
        "company_name": job.company_name,
        "title": job.title,
        "catchphrase": job.catchphrase,
        "salary_type": job.salary_type,
        "salary": job.salary,
        "salary_details": job.salary_details,
        "employment_type": job.employment_type,
        "job_industry": job.job_industry,
        "job_category": job.job_category,
        "social_insurances": job.social_insurances,
        "benefits_1": job.job_benefits_details1,
        "benefits_2": job.job_benefits_details2,
        "holidays_leaves": job.holidays_leaves_details,
        "description": job.description,
        "requirements": job.requirements,
        "requirements_summary": job.requirements_summary,
        "service_form": job.service_form,
        "working_hours": job.working_hours,
        "one_day_work_details": job.one_day_work_details,
        "nearest_station": job.nearest_station,
        "nearest_station_access": job.nearest_station_access,
        "selection_flow": job.selection_flow,
        "recruiter_message": job.recruiter_message,
        "postal_code": job.postal_code,
        "address_details": job.address_details,
        "google_map_url": job.google_map_url,
        "trial_period_duration": job.trial_period_duration,
        "trial_period_details": job.trial_period_details,
        "trial_period_salary": job.trial_period_salary,
        "trial_period_working_hours": job.trial_period_working_hours,
        "tag": job.tag,
    }

    # -------------------------------
    # Identity block (shared context)
    # -------------------------------
    identity = (
        f"Media: {data['media_site']} | "
        f"Company: {data['company_name']} | "
        f"Title: {data['title']}"
    )

    # -------------------------------
    # Tag normalization
    # -------------------------------
    tag_text = ""
    if isinstance(data["tag"], list):
        tag_text = ", ".join(data["tag"])
    elif isinstance(data["tag"], dict):
        tag_text = ", ".join(map(str, data["tag"].values()))
    elif isinstance(data["tag"], str):
        tag_text = data["tag"]

    # -------------------------------
    # Keywords (NO nearest_station)
    # -------------------------------
    keywords = (
        f"Keywords: "
        f"{data['company_name']}, "
        f"{data['job_category']}, "
        f"{data['job_industry']}, "
        f"{data['employment_type']}, "
        f"{tag_text}"
    )

    # -------------------------------
    # Japanese-friendly text splitter
    # -------------------------------
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=600,
        chunk_overlap=100,
        separators=["\n\n", "\n", "。", "！", " ", ""]
    )

    chunks = []

    # =========================================================
    # GROUP 1: OVERVIEW
    # =========================================================
    overview_text = (
        f"{identity}\n"
        f"Catchphrase: {data['catchphrase']}\n"
        f"Industry: {data['job_industry']}\n"
        f"Category: {data['job_category']}\n"
        f"Employment Type: {data['employment_type']}\n"
        f"Service Form: {data['service_form']}\n"
        f"{keywords}"
    )

    chunks.append({
        "text": overview_text,
        "metadata": {
            "serial_no": data["serial_no"],
            "group": "overview"
        }
    })

    # =========================================================
    # GROUP 2: COMPENSATION
    # =========================================================
    compensation_text = (
        f"{identity}\n"
        f"Salary: {data['salary_type']} {data['salary']}\n"
        f"Salary Details: {data['salary_details']}\n"
        f"Social Insurances: {data['social_insurances']}\n"
        f"Benefits: {data['benefits_1']} / {data['benefits_2']}\n"
        f"Holidays: {data['holidays_leaves']}\n"
        f"Trial Period: {data['trial_period_duration']}\n"
        f"Trial Details: {data['trial_period_details']}\n"
        f"Trial Salary: {data['trial_period_salary']}\n"
        f"Trial Working Hours: {data['trial_period_working_hours']}\n"
        f"{keywords}"
    )

    chunks.append({
        "text": compensation_text,
        "metadata": {
            "serial_no": data["serial_no"],
            "group": "compensation"
        }
    })

    # =========================================================
    # GROUP 3: DESCRIPTION (chunked)
    # =========================================================
    if data["description"]:
        desc_parts = text_splitter.split_text(
            f"Job Description:\n{data['description']}\n{keywords}"
        )
        for i, part in enumerate(desc_parts, 1):
            chunks.append({
                "text": f"{identity} | Description Part {i}\n{part}",
                "metadata": {
                    "serial_no": data["serial_no"],
                    "group": f"description_part_{i}"
                }
            })

    # =========================================================
    # GROUP 4: DAILY WORK DETAILS
    # =========================================================
    if data["one_day_work_details"]:
        daily_parts = text_splitter.split_text(
            f"Daily Work Details:\n{data['one_day_work_details']}\n{keywords}"
        )
        for i, part in enumerate(daily_parts, 1):
            chunks.append({
                "text": f"{identity} | Daily Work Part {i}\n{part}",
                "metadata": {
                    "serial_no": data["serial_no"],
                    "group": f"daily_work_part_{i}"
                }
            })

    # =========================================================
    # GROUP 5: REQUIREMENTS + LOGISTICS
    # =========================================================
    logistics_text = (
        f"{identity}\n"
        f"Requirements: {data['requirements']}\n"
        f"Requirements Summary: {data['requirements_summary']}\n"
        f"Working Hours: {data['working_hours']}\n"
        f"Nearest Station: {data['nearest_station']} ({data['nearest_station_access']})\n"
        f"Address: {data['postal_code']} {data['address_details']}\n"
        f"Google Map: {data['google_map_url']}\n"
        f"Selection Flow: {data['selection_flow']}\n"
        f"Recruiter Message: {data['recruiter_message']}\n"
        f"{keywords}"
    )

    chunks.append({
        "text": logistics_text,
        "metadata": {
            "serial_no": data["serial_no"],
            "group": "logistics"
        }
    })

    return chunks






data={
    "serial_no": 2,
    "Media site": "Fudosanworks",
    "Company name": "株式会社バンダイ",
    "Company logo": None,
    "Title": "不動産事務／地域密着・ネイル自由✨にぎやかオフィスの“秘密兵器”募集！",
    "Catchphrase": "髪色・ネイルOKで“自分らしさ”も働きやすさも両方ゲット♪PC作業から外での撮影まで、動きのある事務ワークで営業をサポート◎",
    "Salary type": "月給",
    "Salary": 230000,
    "Salary details": "◇固定残業代について\n◎月給には固定残業代2万円分（月13時間分）を含みます。\n◎固定残業代は残業がない場合も支給し、固定残業時間超過分は別途支給いたします。\n\n◇宅建士資格手当：20,000円",
    "Employment Type": "正社員",
    "Job Industry": "営業・仲介・販売系",
    "Job Category": "営業事務",
    "Social insurances": "◇ 雇用保険\n◇ 厚生年金\n◇ 労災保険\n◇ 健康保険",
    "Job Benefits Details1": "◇ 交通費支給あり\n◇  ネイルOK\n◇ 髪色自由",
    "Job Benefits Details2": None,
    "Holidays & Leaves Details": "◇週休2日（月8休）\n※休み希望日の１ヶ月前申告\n◇ 年末年始休暇\n◇ 夏季休暇\n◇GW休暇\n◇急なお休み考慮OK\n◇慶弔休暇\n◇育児休暇\n◇介護休暇",
    "Description": "＼その“事務観”、たぶんハズレてます…／\n【営業事務】をちょっとだけ想像してみてください…\n\n──「黙々、コツコツ、ひとりで集中…」\nそんな地味なイメージ、ありますよね？\n((ごめんなさい。それ、バンダイでは非対応なんです…))\n\n＼　そもそも、バンダイって何してるの？　／\n2012年に設立し、南浦和・西川口に店舗を展開中♪\n「賃貸」「売買」「管理」「リフォーム」「買取」など、\n 不動産に関することをまるっとサポートしています◎\n地元・埼玉に密着したスタイルで、\n 「住まい」に関する悩みや希望に寄り添う会社です！\n\n\n＼　“バンダイが、ちょっといいかも”と思われる理由　／\n\n≪1.“営業事務”だけど、動きがある！≫\nいわゆる「営業事務」ですが、ずっと座りっぱなしじゃないんです！\n入力や電話対応だけじゃなく、\n物件の写真撮影やちょっとしたお出かけ業務も♪\n\n「今日は事務所でコツコツ集中、明日は少し外に出てリフレッシュ」\nそんなバランスがあるからこそ、飽きずに楽しく続けられます✨\n\n≪2. “やさしさ設計”の働き方、ちゃんとあります≫\n\n髪色・ネイル自由◎「自分らしさ」を大事にできる職場✨\n基本は定時退社だから、オフの時間も楽しめる！\n\n≪3.“ちょうどいい”人間関係が心地いい≫\n\n定期的に社内イベントを開催中♪\nこれまでには、\n ✅ 釣り\n ✅ バーベキュー\n ✅ 中華街で食べ歩きツアー　などなど…！\n仕事中は集中モード、オフはゆるっとリフレッシュ◎\n\nもちろん【参加は自由】だから、\n「お家でのんびり派」な人も気兼ねなくマイペースでOK！\n自然体で関われる、そんな“ちょうどいい距離感”があるのも、バンダイのいいところです♪\n\n＼　具体的な仕事内容　／\n ・物件情報の入力（ポータルサイト・自社HPなど）\n ・お問い合わせ対応（電話・メールの一次対応）\n ・申込手続き（WEBフォーム送信・書類の代筆）\n ・物件の写真撮影、台帳作成などのサポート業務\n→いわゆる“営業サポート”のポジション◎\n 「〇〇さんがいて助かった！」と感謝されることも多いのも魅力✨\n\n未経験でも大丈夫！\nパソコンの基本操作ができれば、事務デビューも大歓迎です◎",
    "Requirements": "【応募資格・条件】\n✅要普通自動車免許（AT限定可）\n✅PC基本操作ができる方\n※Word、Excel入力作業\n\n【歓迎資格】\n✨宅建士資格（手当あり）\n\n・学歴不問\n・未経験OK",
    "Requirements summary": "【応募資格・条件】\n✅要普通自動車免許（AT限定可）\n✅PC基本操作ができる方\n※Word、Excel入力作業\n\n【歓迎資格】\n✨宅建士資格（手当あり）\n\n",
    "Service Form": "出社勤務",
    "Working hours": "9：30～19：00（休憩1時間30分）",
    "One day work details": None,
    "Nearest Station": "南浦和駅\n川口駅",
    "Nearest station access": "JR京浜東北線「南浦和駅」より徒歩5分\nJR武蔵野線「南浦和駅」より徒歩5分\nJR京浜東北線「川口駅」より徒歩5分",
    "Selection flow": "本求人は株式会社ライフアップの採用支援サービスとなります。\nご希望の際には転職活動のサポートもございます。\n（職業紹介事業許可番号：13-ユ-316557）\n　\n【選考の流れ】\n応募⇒オンライン面談⇒面接（1～3回程度）⇒入社",
    "Recruiter message": None,
    "Postal Code": "336-0017",
    "Address details": "埼玉県さいたま市南区南浦和2-32-10",
    "Google Map Url": "https://www.google.com/maps?q=%E5%9F%BC%E7%8E%89%E7%9C%8C%E3%81%95%E3%81%84%E3%81%9F%E3%81%BE%E5%B8%82%E5%8D%97%E5%8C%BA%E5%8D%97%E6%B5%A6%E5%92%8C2-32-10",
    "Trial period duration": "3カ月",
    "Trial period details": "給与・待遇変動なし",
    "Trial period salary": None,
    "Trial period working hours": None,
    "image data": None,
    "Tag": [
      "出社勤務",
      "資格手当あり",
      "未経験歓迎",
      "経験者優遇",
      "第二新卒歓迎",
      "異業種からの転職歓迎",
      "マネジメント経験者歓迎",
      "ネイルOK",
      "髪色自由"
    ]
  }


job = JobInput(**data)

chunks = chunking(job)

for i in range (0, len(chunks)):
    print(f"chunk {i} ===== {chunks[i]}")

chunk 0 ===== {'text': 'Media: Fudosanworks | Company: 株式会社バンダイ | Title: 不動産事務／地域密着・ネイル自由✨にぎやかオフィスの“秘密兵器”募集！\nCatchphrase: None\nIndustry: 営業・仲介・販売系\nCategory: 営業事務\nEmployment Type: 正社員\nService Form: 出社勤務\nKeywords: 株式会社バンダイ, 営業事務, 営業・仲介・販売系, 正社員, 出社勤務, 資格手当あり, 未経験歓迎, 経験者優遇, 第二新卒歓迎, 異業種からの転職歓迎, マネジメント経験者歓迎, ネイルOK, 髪色自由', 'metadata': {'serial_no': 2, 'group': 'overview'}}
chunk 1 ===== {'text': 'Media: Fudosanworks | Company: 株式会社バンダイ | Title: 不動産事務／地域密着・ネイル自由✨にぎやかオフィスの“秘密兵器”募集！\nSalary: 月給 None\nSalary Details: ◇固定残業代について\n◎月給には固定残業代2万円分（月13時間分）を含みます。\n◎固定残業代は残業がない場合も支給し、固定残業時間超過分は別途支給いたします。\n\n◇宅建士資格手当：20,000円\nSocial Insurances: ◇\u2002雇用保険\n◇\u2002厚生年金\n◇\u2002労災保険\n◇\u2002健康保険\nBenefits: ◇\u2002交通費支給あり\n◇\xa0 ネイルOK\n◇ 髪色自由 / None\nHolidays: ◇週休2日（月8休）\n※休み希望日の１ヶ月前申告\n◇\u2002年末年始休暇\n◇\u2002夏季休暇\n◇GW休暇\n◇急なお休み考慮OK\n◇慶弔休暇\n◇育児休暇\n◇介護休暇\nTrial Period: 3カ月\nTrial Details: 給与・待遇変動なし\nTrial Salary: None\nTrial Working Hours: None\nKeywords: 株式会社バンダイ, 営業事務, 営業・仲介・販売系, 正社員, 出社勤務, 資格手当あり, 

/Users/alamin/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [ ]:
chunk 0 ===== {'text': 'Media: Fudosanworks | Company: 株式会社バンダイ | Title: 不動産事務／地域密着・ネイル自由✨にぎやかオフィスの“秘密兵器”募集！\nCatchphrase: None\nIndustry: 営業・仲介・販売系\nCategory: 営業事務\nEmployment Type: 正社員\nService Form: 出社勤務\n
Keywords: 株式会社バンダイ, 営業事務, 営業・仲介・販売系, 正社員, 出社勤務, 資格手当あり, 未経験歓迎, 経験者優遇, 第二新卒歓迎, 異業種からの転職歓迎, マネジメント経験者歓迎, ネイルOK, 髪色自由', 

'metadata': {'serial_no': 2, 'group': 'overview'}}
